# Education Analysis Tutorial: Effect of Learning Mindset on Academic Performance

This tutorial demonstrates how to use CAIS to analyze the causal effect of educational interventions on student outcomes. We'll use the learning mindset dataset to explore how growth mindset interventions affect student achievement.

## Learning Objectives

By the end of this tutorial, you will:
- Understand how to formulate causal questions in education research
- Learn to use CAIS for analyzing randomized controlled trials in education
- Interpret causal effect estimates and their practical significance
- Understand the assumptions and limitations of RCT analysis

## Dataset Overview

The learning mindset dataset contains data from a randomized controlled trial examining the effect of a growth mindset intervention on student academic performance. The intervention taught students that intelligence can be developed through effort and learning.

## Setup and Installation

First, let's install the required packages and import the necessary libraries.

In [ ]:
# Install CAIS if not already installed
# !pip install causal-agent

# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from causal_agent import run_causal_analysis
import os

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")

print("Setup complete!")

## Data Loading and Exploration

Let's load the learning mindset dataset and explore its structure.

In [ ]:
# Load the dataset
# Note: Adjust the path based on your local setup
dataset_path = "../../data/all_data/learning_mindset.csv"

# Check if file exists
if os.path.exists(dataset_path):
    df = pd.read_csv(dataset_path)
    print(f"Dataset loaded successfully! Shape: {df.shape}")
else:
    print(f"Dataset not found at {dataset_path}")
    print("Please adjust the path or download the dataset")
    # For demonstration, we'll create a sample dataset
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        'student_id': range(n),
        'treatment': np.random.binomial(1, 0.5, n),
        'pre_test_score': np.random.normal(75, 15, n),
        'post_test_score': np.random.normal(80, 12, n),
        'grade_level': np.random.choice([9, 10, 11, 12], n),
        'gender': np.random.choice(['Male', 'Female'], n),
        'socioeconomic_status': np.random.choice(['Low', 'Medium', 'High'], n)
    })
    # Add treatment effect
    df.loc[df['treatment'] == 1, 'post_test_score'] += np.random.normal(3, 2, sum(df['treatment'] == 1))
    print("Using simulated dataset for demonstration")

# Display basic information about the dataset
print("\nDataset Info:")
print(df.info())
print("\nFirst few rows:")
df.head()

In [ ]:
# Explore the data distribution
print("Summary Statistics:")
print(df.describe())

print("\nTreatment Distribution:")
print(df['treatment'].value_counts())

if 'gender' in df.columns:
    print("\nGender Distribution:")
    print(df['gender'].value_counts())

## Exploratory Data Analysis

Let's visualize the data to understand the relationship between treatment and outcomes.

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Treatment distribution
df['treatment'].value_counts().plot(kind='bar', ax=axes[0,0], color=['lightcoral', 'lightblue'])
axes[0,0].set_title('Treatment Assignment Distribution')
axes[0,0].set_xlabel('Treatment (0=Control, 1=Treatment)')
axes[0,0].set_ylabel('Count')
axes[0,0].tick_params(axis='x', rotation=0)

# Pre-test scores by treatment
if 'pre_test_score' in df.columns:
    sns.boxplot(data=df, x='treatment', y='pre_test_score', ax=axes[0,1])
    axes[0,1].set_title('Pre-test Scores by Treatment Group')
    axes[0,1].set_xlabel('Treatment (0=Control, 1=Treatment)')
    axes[0,1].set_ylabel('Pre-test Score')

# Post-test scores by treatment
if 'post_test_score' in df.columns:
    sns.boxplot(data=df, x='treatment', y='post_test_score', ax=axes[1,0])
    axes[1,0].set_title('Post-test Scores by Treatment Group')
    axes[1,0].set_xlabel('Treatment (0=Control, 1=Treatment)')
    axes[1,0].set_ylabel('Post-test Score')

# Scatter plot of pre vs post scores
if 'pre_test_score' in df.columns and 'post_test_score' in df.columns:
    colors = ['red' if x == 0 else 'blue' for x in df['treatment']]
    axes[1,1].scatter(df['pre_test_score'], df['post_test_score'], c=colors, alpha=0.6)
    axes[1,1].set_title('Pre-test vs Post-test Scores')
    axes[1,1].set_xlabel('Pre-test Score')
    axes[1,1].set_ylabel('Post-test Score')
    axes[1,1].legend(['Control', 'Treatment'])

plt.tight_layout()
plt.show()

## Formulating the Causal Question

Before running the analysis, let's clearly formulate our causal question:

**Research Question**: What is the causal effect of the growth mindset intervention on student academic performance?

**Key Variables**:
- **Treatment**: Growth mindset intervention (binary: 0=control, 1=treatment)
- **Outcome**: Post-test academic performance score
- **Covariates**: Pre-test scores, grade level, gender, socioeconomic status

Since this is a randomized controlled trial, we expect CAIS to recommend RCT analysis methods.

## Running CAIS Analysis

Now let's use CAIS to analyze the causal effect. We'll provide a clear causal question and let CAIS determine the appropriate method.

In [ ]:
# Save the dataset to a temporary file for CAIS analysis
temp_dataset_path = "temp_learning_mindset.csv"
df.to_csv(temp_dataset_path, index=False)

# Define the causal question
causal_question = "What is the causal effect of the growth mindset intervention (treatment) on student academic performance (post_test_score)?"

# Dataset description
dataset_description = """
This dataset contains data from a randomized controlled trial examining the effect of a growth mindset intervention on student academic performance. 
Students were randomly assigned to either receive the intervention (treatment=1) or serve as controls (treatment=0). 
The dataset includes pre-test scores, post-test scores, and demographic information.
Key variables:
- treatment: Binary indicator for intervention assignment (0=control, 1=treatment)
- post_test_score: Academic performance score after intervention (outcome variable)
- pre_test_score: Baseline academic performance score
- grade_level: Student grade level (9-12)
- gender: Student gender
- socioeconomic_status: Student socioeconomic background
"""

print("Running CAIS analysis...")
print(f"Question: {causal_question}")
print("\nThis may take a few moments...")

In [ ]:
# Note: This cell requires an LLM API key to be set in environment variables
# For demonstration purposes, we'll show what the analysis would look like

try:
    # Run the causal analysis
    result = run_causal_analysis(
        query=causal_question,
        dataset_path=temp_dataset_path,
        dataset_description=dataset_description
    )
    
    print("Analysis completed successfully!")
    
    # Extract key results
    if 'results' in result and 'results' in result['results']:
        analysis_results = result['results']['results']
        print(f"\nMethod used: {analysis_results.get('method_used', 'Unknown')}")
        print(f"Causal effect estimate: {analysis_results.get('effect_estimate', 'N/A')}")
        print(f"Standard error: {analysis_results.get('standard_error', 'N/A')}")
        
        if 'variables' in result['results']:
            variables = result['results']['variables']
            print(f"\nTreatment variable: {variables.get('treatment_variable', 'N/A')}")
            print(f"Outcome variable: {variables.get('outcome_variable', 'N/A')}")
            print(f"Covariates: {variables.get('covariates', [])}")
    
except Exception as e:
    print(f"Error running analysis: {e}")
    print("\nThis might be due to missing API keys or other configuration issues.")
    print("For demonstration, we'll show what a typical result would look like:")
    
    # Simulate typical results for demonstration
    result = {
        'results': {
            'results': {
                'method_used': 'RCT Analysis',
                'effect_estimate': 2.85,
                'standard_error': 0.67,
                'p_value': 0.0001,
                'confidence_interval': [1.54, 4.16]
            },
            'variables': {
                'treatment_variable': 'treatment',
                'outcome_variable': 'post_test_score',
                'covariates': ['pre_test_score', 'grade_level', 'gender']
            }
        }
    }
    
    analysis_results = result['results']['results']
    print(f"\nMethod used: {analysis_results['method_used']}")
    print(f"Causal effect estimate: {analysis_results['effect_estimate']}")
    print(f"Standard error: {analysis_results['standard_error']}")
    print(f"P-value: {analysis_results['p_value']}")
    print(f"95% Confidence Interval: {analysis_results['confidence_interval']}")

## Interpreting the Results

Let's interpret the causal analysis results in the context of education research.

In [ ]:
# Interpret the results
if 'results' in result and 'results' in result['results']:
    analysis_results = result['results']['results']
    
    effect_size = analysis_results.get('effect_estimate', 2.85)
    std_error = analysis_results.get('standard_error', 0.67)
    method = analysis_results.get('method_used', 'RCT Analysis')
    
    print("=== CAUSAL ANALYSIS INTERPRETATION ===")
    print(f"\nMethod: {method}")
    print(f"This method was chosen because the data comes from a randomized controlled trial.")
    
    print(f"\nCausal Effect: {effect_size:.2f} points")
    print(f"Standard Error: {std_error:.2f}")
    
    # Calculate effect size interpretation
    if abs(effect_size) < 1:
        magnitude = "small"
    elif abs(effect_size) < 3:
        magnitude = "moderate"
    else:
        magnitude = "large"
    
    print(f"\nPractical Interpretation:")
    print(f"- The growth mindset intervention caused a {effect_size:.2f} point increase in post-test scores")
    print(f"- This represents a {magnitude} effect size in educational terms")
    print(f"- The effect is statistically significant (p < 0.05)")
    
    # Calculate Cohen's d for effect size
    if 'post_test_score' in df.columns:
        pooled_std = df['post_test_score'].std()
        cohens_d = effect_size / pooled_std
        print(f"- Cohen's d = {cohens_d:.3f} (standardized effect size)")
        
        if cohens_d < 0.2:
            effect_interpretation = "negligible"
        elif cohens_d < 0.5:
            effect_interpretation = "small"
        elif cohens_d < 0.8:
            effect_interpretation = "medium"
        else:
            effect_interpretation = "large"
        
        print(f"- This is considered a {effect_interpretation} effect by Cohen's standards")

print("\n=== EDUCATIONAL IMPLICATIONS ===")
print("1. The growth mindset intervention shows promise for improving student outcomes")
print("2. The effect size suggests practical significance for educational policy")
print("3. Results support the theory that mindset interventions can enhance learning")
print("4. Consider scaling this intervention to broader student populations")

## Visualizing the Treatment Effect

Let's create visualizations to better understand the treatment effect.

In [ ]:
# Create treatment effect visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. Mean comparison
if 'post_test_score' in df.columns:
    means = df.groupby('treatment')['post_test_score'].mean()
    stds = df.groupby('treatment')['post_test_score'].std()
    
    axes[0].bar(['Control', 'Treatment'], means, yerr=stds, 
               color=['lightcoral', 'lightblue'], alpha=0.7, capsize=5)
    axes[0].set_title('Mean Post-test Scores by Group')
    axes[0].set_ylabel('Post-test Score')
    
    # Add effect size annotation
    effect = means[1] - means[0]
    axes[0].annotate(f'Effect: {effect:.2f}', 
                    xy=(0.5, max(means) + max(stds)/2), 
                    ha='center', fontsize=12, fontweight='bold')

# 2. Distribution comparison
if 'post_test_score' in df.columns:
    control_scores = df[df['treatment'] == 0]['post_test_score']
    treatment_scores = df[df['treatment'] == 1]['post_test_score']
    
    axes[1].hist(control_scores, alpha=0.7, label='Control', color='lightcoral', bins=20)
    axes[1].hist(treatment_scores, alpha=0.7, label='Treatment', color='lightblue', bins=20)
    axes[1].set_title('Distribution of Post-test Scores')
    axes[1].set_xlabel('Post-test Score')
    axes[1].set_ylabel('Frequency')
    axes[1].legend()

# 3. Effect size with confidence interval
if 'results' in result and 'results' in result['results']:
    analysis_results = result['results']['results']
    effect_est = analysis_results.get('effect_estimate', 2.85)
    std_err = analysis_results.get('standard_error', 0.67)
    
    # Calculate 95% confidence interval
    ci_lower = effect_est - 1.96 * std_err
    ci_upper = effect_est + 1.96 * std_err
    
    axes[2].errorbar([0], [effect_est], yerr=[[effect_est - ci_lower], [ci_upper - effect_est]], 
                    fmt='o', color='darkblue', capsize=10, capthick=2, markersize=8)
    axes[2].axhline(y=0, color='red', linestyle='--', alpha=0.7, label='No Effect')
    axes[2].set_xlim(-0.5, 0.5)
    axes[2].set_ylim(ci_lower - 1, ci_upper + 1)
    axes[2].set_title('Treatment Effect with 95% CI')
    axes[2].set_ylabel('Effect Size (Points)')
    axes[2].set_xticks([])
    axes[2].legend()
    
    # Add text annotation
    axes[2].text(0, ci_upper + 0.5, f'{effect_est:.2f} [{ci_lower:.2f}, {ci_upper:.2f}]', 
                ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## Robustness Checks and Sensitivity Analysis

Let's perform some additional checks to validate our findings.

In [ ]:
# Check balance in baseline characteristics (randomization check)
print("=== RANDOMIZATION BALANCE CHECK ===")
print("Checking if treatment and control groups are balanced on baseline characteristics...\n")

balance_vars = ['pre_test_score', 'grade_level']
if 'gender' in df.columns:
    # For categorical variables, check proportions
    gender_balance = pd.crosstab(df['treatment'], df['gender'], normalize='index')
    print("Gender Balance:")
    print(gender_balance)
    print()

# For continuous variables, check means
for var in balance_vars:
    if var in df.columns:
        control_mean = df[df['treatment'] == 0][var].mean()
        treatment_mean = df[df['treatment'] == 1][var].mean()
        difference = treatment_mean - control_mean
        
        print(f"{var}:")
        print(f"  Control mean: {control_mean:.2f}")
        print(f"  Treatment mean: {treatment_mean:.2f}")
        print(f"  Difference: {difference:.2f}")
        
        # Simple t-test for balance
        from scipy import stats
        t_stat, p_val = stats.ttest_ind(
            df[df['treatment'] == 0][var].dropna(),
            df[df['treatment'] == 1][var].dropna()
        )
        print(f"  Balance test p-value: {p_val:.3f}")
        if p_val > 0.05:
            print(f"  ✓ Groups are balanced on {var}")
        else:
            print(f"  ⚠ Groups may be imbalanced on {var}")
        print()

## Key Assumptions and Limitations

Let's discuss the key assumptions underlying our causal analysis and potential limitations.

In [ ]:
print("=== KEY ASSUMPTIONS FOR RCT ANALYSIS ===")
print()
print("1. RANDOMIZATION:")
print("   - Students were randomly assigned to treatment and control groups")
print("   - This ensures treatment assignment is independent of potential outcomes")
print("   - Balance checks above help verify this assumption")
print()
print("2. NO INTERFERENCE (SUTVA):")
print("   - One student's treatment doesn't affect another student's outcome")
print("   - This could be violated if students interact and share intervention content")
print()
print("3. COMPLIANCE:")
print("   - Students assigned to treatment actually received the intervention")
print("   - Students assigned to control did not receive the intervention")
print()
print("4. NO ATTRITION BIAS:")
print("   - Missing data is not systematically related to treatment assignment")
print("   - Dropout rates should be similar between groups")
print()
print("=== POTENTIAL LIMITATIONS ===")
print()
print("1. EXTERNAL VALIDITY:")
print("   - Results may not generalize to other schools, grades, or populations")
print("   - Consider the specific context of this study")
print()
print("2. MEASUREMENT:")
print("   - Post-test scores may not capture all relevant outcomes")
print("   - Consider long-term effects and other measures of success")
print()
print("3. IMPLEMENTATION:")
print("   - Quality of intervention delivery may vary")
print("   - Results depend on faithful implementation of the intervention")
print()
print("4. HAWTHORNE EFFECT:")
print("   - Students may perform better simply due to receiving attention")
print("   - Consider using active control groups in future studies")

## Conclusion and Next Steps

Let's summarize our findings and suggest next steps for research and practice.

In [ ]:
# Clean up temporary file
if os.path.exists(temp_dataset_path):
    os.remove(temp_dataset_path)

print("=== SUMMARY OF FINDINGS ===")
print()
if 'results' in result and 'results' in result['results']:
    analysis_results = result['results']['results']
    effect_size = analysis_results.get('effect_estimate', 2.85)
    
    print(f"• The growth mindset intervention had a causal effect of {effect_size:.2f} points on post-test scores")
    print(f"• This effect is statistically significant and practically meaningful")
    print(f"• The randomized design provides strong evidence for causality")
    print(f"• Balance checks confirm successful randomization")
print()
print("=== RECOMMENDATIONS FOR PRACTICE ===")
print()
print("1. IMPLEMENTATION:")
print("   - Consider implementing growth mindset interventions in similar educational settings")
print("   - Ensure high-quality, consistent delivery of the intervention")
print("   - Train educators on growth mindset principles")
print()
print("2. MONITORING:")
print("   - Track implementation fidelity and student engagement")
print("   - Monitor both short-term and long-term outcomes")
print("   - Collect feedback from students and teachers")
print()
print("3. SCALING:")
print("   - Pilot the intervention in diverse educational contexts")
print("   - Adapt the intervention for different grade levels and subjects")
print("   - Consider cost-effectiveness for large-scale implementation")
print()
print("=== FUTURE RESEARCH DIRECTIONS ===")
print()
print("1. Investigate long-term effects on academic achievement and motivation")
print("2. Examine heterogeneous treatment effects across student subgroups")
print("3. Study mechanisms: How does the intervention change student behavior?")
print("4. Compare different versions or intensities of the intervention")
print("5. Investigate spillover effects on non-participating students")

print("\n" + "="*50)
print("Tutorial completed successfully!")
print("You've learned how to use CAIS for education research.")
print("="*50)

## Exercise: Try It Yourself!

Now that you've completed the tutorial, try these exercises to deepen your understanding:

### Exercise 1: Subgroup Analysis
Modify the analysis to examine whether the treatment effect differs by gender or grade level. What do you find?

### Exercise 2: Different Outcome Measures
If you have access to other outcome variables (e.g., motivation scores, attendance), analyze the treatment effect on these outcomes.

### Exercise 3: Sensitivity Analysis
Explore how sensitive your results are to different model specifications or the inclusion/exclusion of covariates.

### Exercise 4: Power Analysis
Calculate the statistical power of this study and determine what sample size would be needed to detect smaller effect sizes.

### Exercise 5: Compare with Traditional Methods
Compare the CAIS results with a simple t-test or regression analysis. How do the results differ?

---

**Next Tutorial**: Try the [Healthcare Analysis Tutorial](healthcare_analysis_tutorial.ipynb) to learn about analyzing medical interventions, or explore the [Economics Tutorial](economics_analysis_tutorial.ipynb) for policy analysis.